# Agricultural Expert System with LLM Enhancement
A domain-specific question-answering system designed to assist Egyptian farmers with crop management, disease diagnosis, and agricultural guidance.

The system is built on a manually curated knowledge base covering **90+ crops** with **60+ parameters per crop**, enhanced with a local LLM layer (Aya) to improve query understanding and response clarity for non-technical users.

In [1]:
import json
import os
import requests

## Loading Knowledge Base
Loading the manually curated expert data and question map from JSON files.
- **ExpertDatajson.json** — domain knowledge covering 90+ crops and 60+ parameters per crop, sourced and structured manually
- **questionmap.json** — maps common question patterns to data fields to improve query understanding

In [2]:
expert_data = json.loads(open('ExpertDatajson.json', encoding='utf-8').read())
question_map_data = json.loads(open('questionmap.json', encoding='utf-8').read())

## Configuration

In [3]:
OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "aya" 

## Prompt bulding

In [38]:
def build_prompt(user_question, expert_knowledge, q_map=None):
    expert_knowledge_string = json.dumps(expert_knowledge, ensure_ascii=False, indent=2)

    q_map_section = ""
    if q_map:
        q_map_string = json.dumps(q_map, ensure_ascii=False, indent=2)
        q_map_section = f"""
دي بعض الأمثلة لطرق سؤال المستخدم عن البيانات، عشان تساعدك تفهم القصد حتى لو السؤال مش حرفي:
{q_map_string}
"""

    prompt = f"""You are an expert agricultural assistant. Follow these rules strictly:

1. Prefer the DATA section below as your primary source. If the answer is fully or partially there, use it.
2. If the DATA doesn't cover the question (or only partly covers it), you may use your own general agricultural knowledge to complete the answer.
3. Understand the user's intent even if their question is phrased differently from the data fields.
4. Be concise and direct: answer in short points, only what was asked, no extra unrelated details, no long explanations.
5. Do not repeat the question or add introductions/conclusions — go straight to the answer.
6. Write your final answer in Egyptian colloquial Arabic (مصري عامي), professional and clear.

DATA:
{expert_knowledge_string}

{q_map_section}

User question (in Arabic): "{user_question}"

Now respond in Egyptian Arabic, in short direct points:
"""
    return prompt

## LLM Response Generator

In [34]:
def generate_response(prompt_text):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "prompt": prompt_text,
            "stream": False,
            "num_predict": 500
        }
    )
    return response.json()['response']

## Questions function and Testing

In [35]:
def ask(question):
    prompt = build_prompt(question, expert_data, q_map=question_map_data)
    response = generate_response(prompt)
    print(f"السؤال: {question}")
    print(f"{response}\n")

In [39]:
ask("ازاي ازرع الطماطم وابيعها")

السؤال: ازاي ازرع الطماطم وابيعها
- الطماطم ممكن تزرع في أي وقت خلال السنة، لكن أفضل وقت هو عندما تكون درجة الحرارة بتناسب النمو. 
- الفاصل بين الزراعة والحصاد ممكن يتراوح من أسبوعين لثلاثة أشهر حسب نوع الطماطم.
- طول دورة حياة المحصول ممكن يختلف حسب نوع الطماطم، لكن في المتوسط هيأخذ حوالي ثلاثة شهور.
- الوقت اللي بيستغرقه النبات لينضج ممكن يتغير حسب النوع ودرجة الحرارة، لكن في المتوسط هياخد حوالي 90 يوم. 
- عدد الأيام بين الزراعة والحصاد ممكن يتراوح من أسبوعين لثلاثة أشهر.
- إنتاجية الفدان ممكن تختلف حسب نوع الطماطم وطريقة الزراعة، لكن في المتوسط هيأخذ من شهر ونصف لشهرين.
- تخزين الطماطم بعد الحصاد ممكن يناسب:
 - التخزين في الأماكن الباردة (مثل الثلاجة أو الفريزر) لمدة تصل إلى 3 شهور. 
 - تخزينها في أماكن جافة وبعيدة عن ضوء الشمس المباشر لمدة تصل إلى شهرين. 
 - التخزين في صناديق مغلقة وجافة لمدة تصل إلى شهر واحد. 
- درجة حرارة التخزين المثالية للطماطم هي من 2° الى 15°.
- بعد الحصاد، الطماطم ممكن يبقى سليم لمدة تصل إلى شهرين. 
- التغليف المناسب للطماطم:
 - يجب أن يكون مغلف معتم وليكن ر

In [15]:
ask("ايه انسب وقت ازرع فيه الكرمب")

السؤال: ايه انسب وقت ازرع فيه الكرمب
الإجابة: بالتأكيد! فيما يلي إجابات على أسئلتك من القوائم المحددة:

- "الفترة المثالية لزراعة الكرمب": يفضل زراعة الكرمب خلال فصل الربيع، وتحديداً في أواخر شهر مارس أو أوائل شهر أبريل. فهذا الوقت يوفر للمحصول فترة نمو كافية قبل حلول الحرارة الشديدة في الصيف.

- "المدة بين الزراعة والحصاد": تختلف المدة حسب طرق الزراعة وظروف النمو، ولكن عادة ما يستغرق الكرمب حوالي 7-9 أشهر من وقت الزراعة حتى الحصاد. فبعد الزراعة مباشرة، ينمو النبات بسرعة ويبدأ في توليد السيقان والأوراق. ثم بعد ذلك، يتطور النبات إلى شكل الكرمش، ويستغرق الأمر حوالي شهرين إلى ثلاثة أشهر حتى يصبح ناضجاً للحصاد.

- "إنتاجية الفدان": يمكن أن تتراوح إنتاجية الفدان من الكرمب بين 5 إلى 10 أطنان، ولكن هذا يعتمد على عوامل مختلفة مثل نوع المحصول وظروف التربة والري والتسميد. فعلى سبيل المثال، قد ينتج نوع معين من الكرمب 7 أطنان للفدان، في حين أن نوعاً آخر قد ينتج 12 طناً.

- "التخزين السليم": للتخزين السليم للكرمب، يجب إتباع النصائح التالية:

   - يجب حصاد الكرمب عندما يكون ناضجاً تماماً، مع التأكد 

In [16]:
ask("ايه هي الامراض الي ممكن تقابل البطاطس")

السؤال: ايه هي الامراض الي ممكن تقابل البطاطس
الإجابة: بالتأكيد، يمكنني مساعدتك في ذلك. فيما يلي إجابات على أسئلتك بناءً على المعلومات المتوفرة في قاعدة البيانات:

- "ما هي الأمراض التي قد تصيب نبات البطاطس؟":
   - تعفن الجذور: ينتج عن تلف الجذور بسبب الفطريات أو البكتيريا، مما يؤدي إلى تعفنها وذبول النبات.
   - آفات الحشرات: يمكن أن تتعرض نباتات البطاطس لهجمات الحشرات مثل المن والعدوة والقراد.
   - أمراض فطرية: قد تصاب نباتات البطاطس بأمراض فطرية مثل البياض أو العفن الفطري.
   - نقص المغذيات: قد يؤدي نقص البوتاسيوم أو النيتروجين إلى ضعف نمو النبات وهشاشته للتلف.
   - الإصابة بالبكتيريا: يمكن أن تتعرض البطاطس لعدة بكتيريا مثل بكتيريا "الدراق" التي تسبب بقعًا بنية على الأوراق والجذور.

- "ما هي أعراض الإصابة بأمراض البطاطس؟":
   - التعفن الجذر: ظهور رائحة كريهة، وذبول النبات، وجفاف الأوراق.
   - آفات الحشرات: وجود ثقوب في أوراق النبات، وحشرات حية أو برازها على الأوراق والجذور.
   - الأمراض الفطرية: ظهور بقع بيضاء أو رمادية على الأوراق أو الجذور، وفقدان النمو الصحي.
   - نقص المغذيات: اص

In [37]:
ask("ازاي ااقدر اعالج العفن الاسود في شجر التفاح")

السؤال: ازاي ااقدر اعالج العفن الاسود في شجر التفاح
- العفن الأسود في أشجار التفاح يظهر بسبب الرطوبة الزائدة أو سوء التهوية.  
- يجب عليك تشذيب الأشجار وإزالة الفروع الميتة وأوراق الشجر لزيادة تهوية الشجرة والحد من الإصابة بالعفونة السوداء.  
- يمكنك استخدام محلول فطري مثل البيريثرويد  أو الكبريت الميكروبي في رش أوراق الشجر والأغصان لمنع انتشار العفن الأسود.  
- تجنب الري الزائد واستخدم أسمدة قليلة النيتروجين لتجفيف النبات.  
- قم برش الشجرة بمحلول فطري بعد الزراعة مباشرةً لمنع الإصابة بالعفونة السوداء في المستقبل.

